Here we will create 2 gold layer entities:
- We will start by creating a view in the gold layer against our silver table customers_orders
- 

In [0]:
%run ../Includes/Copy-Datasets

In [0]:
%sql
CREATE VIEW IF NOT EXISTS countries_stats_vw AS 
(
    SELECT country, date_trunc("DD", order_timestamp) order_date, count(order_id) orders_count, sum(quantity) books_count
    from customers_orders
    group by country, date_trunc("DD", order_timestamp)
)

In [0]:
%sql
select * from countries_stats_vw limit 5;

In [0]:
%sql
select * from countries_stats_vw where country= 'France';

In [0]:
from pyspark.sql import functions as F

query = (spark.readStream
                 .table("books_sales")
                 .withWatermark("order_timestamp", "10 minutes")
                 .groupBy(
                     F.window("order_timestamp", "5 minutes").alias("time"),
                     "author")
                 .agg(
                     F.count("order_id").alias("orders_count"),
                     F.avg("quantity").alias ("avg_quantity"))
              .writeStream
                 .option("checkpointLocation", f"{bookstore.checkpoint_path}/authors_stats")
                 .trigger(availableNow=True)
                 .table("authors_stats")
            )

query.awaitTermination()

In [0]:
%sql
select * from authors_stats;